# 2. Historical Wind Reliability Analysis

## 2.1 First-Principles Objective
Wind is an intermittent energy source. The grid must instantly match supply and electricity demand. If we solely rely on wind, a wind lull (so called *Dunkelflaute*) could cause blackouts. 

The analytical question is: **How many MW of wind power can we *reliably* expect to be available?**

### Defining "Reliably"
In power systems engineering, relying on an intermittent source requires establishing a confidence interval. A **p95** or **p99** reliability metric means that X MW of power is guaranteed to be available 95% or 99% of the time throughout the year.

## 2.2 Assumptions & Methodology 
1. **Stationarity**: We assume the distribution of wind generation in Jan 2025 is broadly representative, though optimally a full 12-month dataset should be used to account for seasonality.
2. **Capacity vs Outturn**: We analyze the actual generation outturn, not installed capacity. 
3. **The 5th Percentile Rule**: We will define "Reliable Firm Capacity" as the 5th percentile (P05) of generation. This means $95\%$ of the time, the wind generation is *at least* this amount.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
try:
    actuals = pd.read_csv('actuals.csv')
    actuals['targetTime'] = pd.to_datetime(actuals['targetTime'])
except FileNotFoundError:
    print("actuals.csv not found.")

## 2.3 Empirical Cumulative Distribution Function (ECDF)
An ECDF will intuitively show us the percentage of time wind generates less than X amount of power.

In [ ]:
def plot_reliability_curve(actuals):
    gen = actuals['actualGeneration'].dropna().values
    x = np.sort(gen)
    y = np.arange(1, len(x) + 1) / len(x)
    
    p05 = np.percentile(x, 5)
    p50 = np.percentile(x, 50)
    
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, marker='.', linestyle='none', color='royalblue')
    plt.axvline(p05, color='red', linestyle='--', label=f"5th Percentile (P05): {p05:.0f} MW")
    plt.axvline(p50, color='orange', linestyle='--', label=f"Median (P50): {p50:.0f} MW")
    
    plt.title("Cumulative Distribution of Wind Generation")
    plt.xlabel("Generation (MW)")
    plt.ylabel("Cumulative Probability")
    plt.legend()
    plt.show()
    
    return p05, p50

## 2.4 Final Recommendation & Reasoning

Based on the empirical distribution of wind generation:

**Recommendation:** We can reliably expect the **5th percentile (P05) MW** of wind power to be available to meet base electricity demand.

### Why not Mean or Median?
Relying on the mean or median (P50) is catastrophically dangerous for power grids. If we plan for the median wind output, the grid will experience a catastrophic supply deficit 50% of the time. Electricity demands continuous, uninterrupted supply. 

### Trade-offs
- By relying only on the P05 firm capacity layer from wind, we implicitly accept that the remaining demand MUST be met by dispatchable baseline power (Nuclear, Gas, interconnectors, or utility-scale batteries).
- When wind exceeds the P05 threshold (which it will 95% of the time), the "excess" wind is not wasted; it simply displaces the more expensive variable cost fossil fuels from the grid merit order. 
- Ultimately, wind is a fantastic fuel-saver and carbon-reducer, but without multi-day energy storage capabilities, its **reliable 'firm base' capacity** is limited strictly to its statistical minimums during prevailing calm weather systems.